# Fine-tune YOLO on Tooth Detection (PyTorch Lightning + W&B)

This notebook fine-tunes a pretrained YOLOv5 model on the tooth detection dataset using:
- Existing project data-preparation modules (`detection_pipeline`)
- A custom PyTorch Lightning training loop
- Weights & Biases logging for loss and mAP metrics

In [11]:
# If needed, uncomment to install dependencies in the active kernel.
# %pip install -q lightning wandb torchmetrics ultralytics

In [ ]:
from __future__ import annotations

import os
import sys
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from records import split_grouped_records

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger

from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Ensure project imports work from notebook location.
PROJECT_ROOT = Path('/work')
SCRIPTS_ROOT = PROJECT_ROOT / 'scripts'
for p in (PROJECT_ROOT, SCRIPTS_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import config
from detection_pipeline import (
    DetectionDownloadConfig,
    ToothDetectionDataset,
    AugmentedToothDetectionDataset,
    build_detection_records,
    build_detection_train_pipeline,
    load_or_download_detection_dataset,
)

In [13]:
@dataclass
class TrainConfig:
    image_size: int = 640
    batch_size: int = 32
    # Use single-process loading by default to avoid /dev/shm bus errors in containers.
    num_workers: int = 0
    max_epochs: int = 20
    lr: float = 1e-3
    weight_decay: float = 5e-4
    momentum: float = 0.937
    conf_threshold: float = 0.001
    iou_threshold: float = 0.6
    pretrained_weights: str = 'yolov5s.pt'
    wandb_project: str = 'tooth-detection-yolo-lightning'
    wandb_run_name: str = 'yolov5s-finetune'
    force_download: bool = False
    # Set to None to use the full dataset.
    train_subset_size: int | None = None
    val_subset_size: int | None = None

cfg = TrainConfig()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cuda


In [14]:
# Clone YOLOv5 if missing and put it first in import order for yolo-specific utils.
YOLOV5_DIR = PROJECT_ROOT / 'external' / 'yolov5'
YOLOV5_DIR.parent.mkdir(parents=True, exist_ok=True)

if not YOLOV5_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', 'https://github.com/ultralytics/yolov5.git', str(YOLOV5_DIR)
    ], check=True)

if str(YOLOV5_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOV5_DIR))

from models.yolo import Model
from utils.loss import ComputeLoss
from utils.general import intersect_dicts, non_max_suppression

print('YOLOv5 code loaded from:', YOLOV5_DIR)

YOLOv5 code loaded from: /work/external/yolov5


In [ ]:
class YoloTargetAdapterDataset(Dataset):
    """Wraps ToothDetectionDataset to emit labels compatible with YOLOv5 training loss."""

    def __init__(self, base_dataset: Dataset):
        self.base_dataset = base_dataset

    def __len__(self) -> int:
        return len(self.base_dataset)

    def __getitem__(self, index: int):
        image, target = self.base_dataset[index]
        return image, target


def yolo_collate_fn(batch: list[tuple[torch.Tensor, dict[str, Any]]]):
    images = []
    yolo_targets = []
    metric_targets = []

    for i, (img, tgt) in enumerate(batch):
        images.append(img.float())

        boxes_xyxy = tgt['boxes'].float()
        labels_one_based = tgt['labels'].long()

        # TorchMetrics expects class IDs starting at 0.
        metric_targets.append({
            'boxes': boxes_xyxy,
            'labels': (labels_one_based - 1).clamp_min(0),
        })

        if boxes_xyxy.numel() == 0:
            continue

        _, h, w = img.shape
        x1, y1, x2, y2 = boxes_xyxy[:, 0], boxes_xyxy[:, 1], boxes_xyxy[:, 2], boxes_xyxy[:, 3]

        cx = ((x1 + x2) * 0.5) / w
        cy = ((y1 + y2) * 0.5) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h

        cls = (labels_one_based - 1).float().clamp_min(0)
        batch_idx = torch.full((boxes_xyxy.shape[0],), float(i), dtype=torch.float32)

        packed = torch.stack([batch_idx, cls, cx, cy, bw, bh], dim=1)
        yolo_targets.append(packed)

    images = torch.stack(images, dim=0)
    if yolo_targets:
        yolo_targets = torch.cat(yolo_targets, dim=0)
    else:
        yolo_targets = torch.zeros((0, 6), dtype=torch.float32)

    return images, yolo_targets, metric_targets


class ToothDetectionDataModule(L.LightningDataModule):
    def __init__(self, cfg: TrainConfig):
        super().__init__()
        self.cfg = cfg
        self.train_ds = None
        self.val_ds = None
        self.test_ds = None

    def setup(self, stage: str | None = None):
        _, coco_data, image_dirs = load_or_download_detection_dataset(
            DetectionDownloadConfig(api_key=os.getenv('ROBOFLOW_API_KEY')),
            force_download=self.cfg.force_download,
        )
        records, _ = build_detection_records(coco_data, image_dirs)

        train_rec, val_rec, test_rec = split_grouped_records(
            records,
            train_size=config.TRAIN_RATIO,
            val_size=config.VAL_RATIO,
            test_size=config.TEST_RATIO,
            random_state=42
        )

        if self.cfg.train_subset_size is not None:
            train_rec = train_rec[: self.cfg.train_subset_size]
        if self.cfg.val_subset_size is not None:
            val_rec = val_rec[: self.cfg.val_subset_size]

        base_train = ToothDetectionDataset(
            train_rec, image_size=self.cfg.image_size, output_channels=3
        )
        aug_train = AugmentedToothDetectionDataset(base_train, build_detection_train_pipeline())
        
        base_val = ToothDetectionDataset(
            val_rec, image_size=self.cfg.image_size, output_channels=3
        )
        base_test = ToothDetectionDataset(
            test_rec, image_size=self.cfg.image_size, output_channels=3
        )

        self.train_ds = YoloTargetAdapterDataset(aug_train)
        self.val_ds = YoloTargetAdapterDataset(base_val)
        self.test_ds = YoloTargetAdapterDataset(base_test)

        print(f'Train/Val/Test sizes: {len(self.train_ds)}, {len(self.val_ds)}, {len(self.test_ds)}')

    def _loader_kwargs(self) -> dict[str, Any]:
        num_workers = int(self.cfg.num_workers)
        kwargs: dict[str, Any] = {
            'num_workers': num_workers,
            'pin_memory': torch.cuda.is_available(),
            'collate_fn': yolo_collate_fn,
        }
        if num_workers > 0:
            kwargs['persistent_workers'] = True
            kwargs['prefetch_factor'] = 2
        return kwargs

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.cfg.batch_size,
            shuffle=True,
            drop_last=True,
            **self._loader_kwargs(),
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            **self._loader_kwargs(),
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_ds,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            **self._loader_kwargs(),
        )

In [16]:
class LitYOLOv5(L.LightningModule):
    def __init__(self, cfg: TrainConfig, num_classes: int = 32):
        super().__init__()
        self.save_hyperparameters(ignore=["cfg"])

        self.cfg = cfg
        self.num_classes = num_classes
        self.compute_loss = None

        ckpt_path = Path(cfg.pretrained_weights)
        ckpt = torch.load(ckpt_path, map_location="cpu")

        yolo_cfg = ckpt["model"].yaml
        self.model = Model(yolo_cfg, ch=3, nc=num_classes).float()

        pretrained_state = ckpt["model"].float().state_dict()
        compatible_state = intersect_dicts(
            pretrained_state,
            self.model.state_dict(),
            exclude=["anchor"],
        )
        self.model.load_state_dict(compatible_state, strict=False)

        self.model.hyp = {
            "box": 0.05,
            "cls": 0.3,
            "obj": 0.7,
            "cls_pw": 1.0,
            "obj_pw": 1.0,
            "fl_gamma": 0.0,
            "label_smoothing": 0.0,
            "anchor_t": 4.0,
        }

        self.map_metric = MeanAveragePrecision(
            box_format="xyxy",
            class_metrics=False,
        )

    def on_train_start(self):
        self.model.to(self.device)
        self.compute_loss = ComputeLoss(self.model)
        self._fix_compute_loss_device()

    def _fix_compute_loss_device(self):
        if self.compute_loss is None:
            return

        self.compute_loss.device = self.device

        for name, value in vars(self.compute_loss).items():
            if torch.is_tensor(value):
                setattr(self.compute_loss, name, value.to(self.device))

            elif isinstance(value, list):
                setattr(
                    self.compute_loss,
                    name,
                    [
                        item.to(self.device) if torch.is_tensor(item) else item
                        for item in value
                    ],
                )

            elif isinstance(value, tuple):
                setattr(
                    self.compute_loss,
                    name,
                    tuple(
                        item.to(self.device) if torch.is_tensor(item) else item
                        for item in value
                    ),
                )

        if hasattr(self.compute_loss, "anchors"):
            self.compute_loss.anchors = self.compute_loss.anchors.to(self.device)

    def forward(self, x: torch.Tensor):
        return self.model(x)

    def training_step(self, batch, batch_idx: int):
        images, yolo_targets, _ = batch

        yolo_targets = yolo_targets.to(images.device, non_blocking=True)

        if self.compute_loss is None:
            self.compute_loss = ComputeLoss(self.model)

        self._fix_compute_loss_device()

        preds = self.model(images)
        loss, loss_items = self.compute_loss(preds, yolo_targets)

        self.log(
            "train/loss",
            loss,
            prog_bar=True,
            on_step=True,
            on_epoch=True,
            batch_size=images.size(0),
        )
        self.log(
            "train/loss_box",
            loss_items[0],
            on_step=True,
            on_epoch=True,
            batch_size=images.size(0),
        )
        self.log(
            "train/loss_obj",
            loss_items[1],
            on_step=True,
            on_epoch=True,
            batch_size=images.size(0),
        )
        self.log(
            "train/loss_cls",
            loss_items[2],
            on_step=True,
            on_epoch=True,
            batch_size=images.size(0),
        )

        return loss

    def validation_step(self, batch, batch_idx: int):
        images, _, metric_targets = batch

        raw_output = self.model(images)

        if isinstance(raw_output, (tuple, list)):
            preds = raw_output[0]
        else:
            preds = raw_output

        nms_preds = non_max_suppression(
            preds,
            conf_thres=self.cfg.conf_threshold,
            iou_thres=self.cfg.iou_threshold,
            multi_label=False,
            max_det=300,
        )

        metric_preds = []

        for det in nms_preds:
            if det is None or len(det) == 0:
                metric_preds.append(
                    {
                        "boxes": torch.zeros((0, 4), device=self.device),
                        "scores": torch.zeros((0,), device=self.device),
                        "labels": torch.zeros(
                            (0,),
                            dtype=torch.long,
                            device=self.device,
                        ),
                    }
                )
                continue

            metric_preds.append(
                {
                    "boxes": det[:, :4],
                    "scores": det[:, 4],
                    "labels": det[:, 5].long(),
                }
            )

        metric_targets = [
            {
                "boxes": target["boxes"].to(self.device),
                "labels": target["labels"].to(self.device),
            }
            for target in metric_targets
        ]

        self.map_metric.update(metric_preds, metric_targets)

    def on_validation_epoch_end(self):
        metrics = self.map_metric.compute()

        self.log("val/map", metrics["map"], prog_bar=True)
        self.log("val/map_50", metrics["map_50"], prog_bar=True)
        self.log("val/map_75", metrics["map_75"])

        self.map_metric.reset()

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(
            self.model.parameters(),
            lr=self.cfg.lr,
            momentum=self.cfg.momentum,
            weight_decay=self.cfg.weight_decay,
            nesterov=True,
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.cfg.max_epochs,
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "frequency": 1,
            },
        }

In [17]:
# Download pretrained YOLOv5 weights if missing.
weights_path = Path(cfg.pretrained_weights)
if not weights_path.exists():
    import urllib.request
    url = 'https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s.pt'
    urllib.request.urlretrieve(url, str(weights_path))

print('Using weights:', weights_path.resolve())

Using weights: /work/scripts/detection_pipeline/training/yolov5s.pt


In [18]:
from pathlib import Path
import os

import wandb
from dotenv import load_dotenv
from wandb.errors import AuthenticationError

env_path = Path('/work/.env')
if not env_path.exists():
    raise FileNotFoundError(f'.env file not found at {env_path}')

load_dotenv(env_path, override=True)

wandb_secret = (os.getenv('WANDB_API_KEY') or '').strip().strip('"').strip("'")
if not wandb_secret:
    raise RuntimeError(f'WANDB_API_KEY not found in {env_path}')

if len(wandb_secret) < 20:
    raise RuntimeError(
        f'Invalid WANDB_API_KEY length ({len(wandb_secret)}). '
        'Expected a classic API key or a wandb_v1 access token from https://wandb.ai/authorize.'
    )

# Support both classic API keys and wandb_v1 access tokens.
try:
    os.environ['WANDB_API_KEY'] = wandb_secret
    wandb.login(key=wandb_secret, relogin=True)
except AuthenticationError:
    if wandb_secret.startswith('wandb_v1_'):
        # Compatibility fallback for SDKs that validate only classic key shapes.
        compat_key = wandb_secret.replace('wandb_v1_', '', 1)
        os.environ['WANDB_API_KEY'] = compat_key
        wandb.login(key=compat_key, relogin=True)
    else:
        raise

print('W&B login successful using WANDB_API_KEY from .env')

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


W&B login successful using WANDB_API_KEY from .env


In [19]:

datamodule = ToothDetectionDataModule(cfg)
model = LitYOLOv5(cfg=cfg, num_classes=32)

wandb_logger = WandbLogger(
    project=cfg.wandb_project,
    name=cfg.wandb_run_name,
    log_model=True,
)

checkpoint_cb = ModelCheckpoint(
    dirpath=str(PROJECT_ROOT / 'output' / 'checkpoints'),
    filename='yolov5-tooth-{epoch:02d}-{val_map:.4f}',
    monitor='val/map',
    mode='max',
    save_top_k=2,
    save_last=True,
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

trainer = L.Trainer(
    max_epochs=cfg.max_epochs,
    accelerator='auto',
    devices=1,
    precision='16-mixed' if torch.cuda.is_available() else 32,
    logger=wandb_logger,
    callbacks=[checkpoint_cb, lr_monitor],
    log_every_n_steps=10,
)

trainer.fit(model, datamodule=datamodule)

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
Overriding model.yaml nc=80 with nc=32

                 from  n    params  module                                  arguments                   

Train/Val/Test sizes: 4361, 1086, 940


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type                 ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ DetectionModel       │  7.1 M │ train │     0 │
│ 1 │ map_metric │ MeanAveragePrecision │      0 │ train │     0 │
└───┴────────────┴──────────────────────┴────────┴───────┴───────┘

Trainable params: 7.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 7.1 M                                                                                                
Total estimated model params size (MB): 28                                                                         
Modules in train mode: 215                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7a1378501a50>> (for post_run_cell), with arguments args (<ExecutionResult object at 7a129fde79a0, execution_count=19 error_before_exec=None error_in_exec=1 info=<ExecutionInfo object at 7a129fde60e0, raw_cell="
datamodule = ToothDetectionDataModule(cfg)
model .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://attached-container%2B7b22636f6e7461696e65724e616d65223a222f70726f6a6563742d6170702d31222c2273657474696e6773223a7b22636f6e74657874223a226465736b746f702d6c696e7578227d7d/work/scripts/detection_pipeline/training/yolo_lightning_finetune.ipynb#X12sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: [Errno 104] Connection reset by peer

In [ ]:
print('Best checkpoint:', checkpoint_cb.best_model_path)
trainer.validate(model, datamodule=datamodule)

Best checkpoint: /work/output/checkpoints/yolov5-tooth-epoch=19-val_map=0.0000.ckpt


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Train/Val/Test sizes: 100, 10, 940


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val/map          │   0.0041678608395159245   │
│        val/map_50         │    0.01862182281911373    │
│        val/map_75         │            0.0            │
└───────────────────────────┴───────────────────────────┘

[{'val/map': 0.0041678608395159245,
  'val/map_50': 0.01862182281911373,
  'val/map_75': 0.0}]

wandb: WARNING Artifact "model-1dkn2jtf" already exists with the same content. No new version will be created.


## Notes

- The dataset labels are converted from 1..32 to 0..31 for YOLO class indexing.
- Training and validation mAP are logged to W&B through Lightning's logger.
- You can adjust `TrainConfig` to tune learning rate, batch size, and epochs.